In [ ]:
# ================================================================
# NOTEBOOK : nb_gold_store_perf
# Reads    : silver_lakehouse → silver_sales, silver_store
# Writes   : gold_lakehouse  → gold_store_performance
# Logic    : Store-level KPIs — total, MTD, ranking
# ================================================================

In [1]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, round
from pyspark.sql.window import Window

sales = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_sales")
store = spark.sql("SELECT * FROM silver_lakehouse.dbo.silver_store")

StatementMeta(, b63721e3-a3fc-4f9a-a092-bcd8de9e8fb0, 3, Finished, Available, Finished, False)

##### ── Aggregate at store grain (all-time, in this dataset's range) ──

In [10]:
store_agg = ( 
sales.filter(col("IsReturn") == False)
.groupBy("StoreID")
.agg(
    F.sum("TotalAmount").alias("TotalRevenue"),
F.count("TransactionID").alias("TotalTransactions"),
F.avg("TotalAmount").alias("AvgBasketSize"),
F.countDistinct("ProductID").alias("ProductsSold"),
F.count_distinct("TransactionDate").alias("ActiveDays"),
)

)
# ── Join store dimension ──────────────────────────────────────────

gold_df = store_agg.join(store, on="StoreID", how="left")

# ── Rank stores by revenue within their region ────────────────────

region_window = Window.partitionBy("Region").orderBy(F.desc("TotalRevenue"))
overall_window = Window.orderBy(F.desc("TotalRevenue"))



gold_df = (
    gold_df.withColumn("region_window",F.rank().over(region_window))
    .withColumn("OverallRank", F.rank().over(overall_window))
    .withColumn("RevenuePerSqFt", round(col("TotalRevenue"))/col("SquareFootage"))
    .withColumn("_gold_load_ts", F.current_timestamp())
    )
gold_df.write.format("delta")\
.mode("overwrite").option("overwriteSchema", True)\
.saveAsTable("gold_store_performance")


print(f"[DONE] gold_store_performance:{gold_df.count()} rows")
display(gold_df.orderBy("OverallRank").limit(10))

StatementMeta(, b63721e3-a3fc-4f9a-a092-bcd8de9e8fb0, 12, Finished, Available, Finished, False)

[DONE] gold_store_performance:50 rows


SynapseWidget(Synapse.DataFrame, 4b785356-9962-4312-8cd0-c0b624bfacc6)